# JS15 – Pipeline Machine Learning & Deployment
## Tugas Praktikum: Menerapkan End-to-End Pipeline & Deployment pada Studi Kasus Proyek

---

### Deskripsi Tugas
Setelah mempraktikkan proses ekspor model dan deployment pada studi kasus latihan ("Day vs Night"), mahasiswa diinstruksikan untuk menerapkan **pipeline deployment yang sama** untuk model Machine Learning proyek masing-masing:
1. **Penyimpanan Artefak Model**:
   - File Model (`.h5`): Model neural network yang telah dilatih dengan metrik terbaik.
   - File Preprocessor / Scaler (`.pkl`): Objek standarisasi / pipeline data.
2. **Pembuatan File Web Server (`app.py`)**:
   - Memuat model dan scaler.
   - Menyediakan antarmuka input formulir web (HTML GUI).
   - Menjalankan inferensi dan mengembalikan hasil prediksi.
3. **Konfigurasi Lingkungan Deployment**:
   - `requirements.txt`: Daftar pustaka Python minimal yang dibutuhkan server.
   - `Dockerfile`: Konfigurasi container Linux Debian/Python 3.9 slim dengan user non-root dan expose port 7860.
4. **Alur Deployment ke Cloud (Hugging Face Spaces)**:
   - Prosedur Git clone, penambahan berkas, commit, dan push ke server Hugging Face Spaces.


### 1. Struktur Direktori Proyek Deployment Standar
Berikut adalah arsitektur direktori standar yang disiapkan untuk Hugging Face Spaces:

```text
daynight-project/
├── app.py                # Web server Flask untuk inferensi model
├── Dockerfile            # Blueprint container Docker
├── requirements.txt      # Dependensi pustaka
├── day_night_model.h5    # Artefak model neural network terlatih
├── scaler.pkl            # Artefak objek normalisasi fitur
└── README.md             # Panduan deployment & dokumentasi space
```


### 2. Kode Server Web (`app.py`)

In [1]:
import os
# Verifikasi dan Tinjau Kode app.py yang ada di direktori daynight-project
app_path = os.path.join("daynight-project", "app.py")
with open(app_path, "r", encoding="utf-8") as f:
    app_code = f.read()

print(f"File app.py siap dideploy (Panjang kode: {len(app_code)} karakter)")
print("Contoh cuplikan kode app.py:")
print("\n".join(app_code.splitlines()[:25]))


File app.py siap dideploy (Panjang kode: 4207 karakter)
Contoh cuplikan kode app.py:
import os
import numpy as np
import cv2
import pickle
import tensorflow as tf
from flask import Flask, request, render_template_string
from skimage.feature import hog

app = Flask(__name__)

# Load Model & Scaler
MODEL_PATH = 'day_night_model.h5'
SCALER_PATH = 'scaler.pkl'

try:
    model = tf.keras.models.load_model(MODEL_PATH)
    with open(SCALER_PATH, 'rb') as f:
        scaler = pickle.load(f)
    print("✅ System Loaded Successfully")
except Exception as e:
    print(f"❌ Error loading system: {e}")

def preprocess_image(image_bytes):
    nparr = np.frombuffer(image_bytes, np.uint8)
    img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)


### 3. Konfigurasi Container (`Dockerfile`)
Dockerfile menyusun sistem operasi Debian Python 3.9-slim, dependensi grafis OpenCV, hak akses keamanan non-root standar Hugging Face, dan port 7860:


In [2]:
docker_path = os.path.join("daynight-project", "Dockerfile")
with open(docker_path, "r", encoding="utf-8") as f:
    docker_content = f.read()

print("Konfigurasi Dockerfile valid:")
print(docker_content)


Konfigurasi Dockerfile valid:
# Base Image Python 3.9 Slim
FROM python:3.9-slim

# Install Library Sistem untuk OpenCV
RUN apt-get update && apt-get install -y \
    libgl1 \
    libglib2.0-0 \
    && rm -rf /var/lib/apt/lists/*

# Setup User Non-Root (Standar Keamanan Hugging Face Spaces)
RUN useradd -m -u 1000 user
USER user
ENV PATH="/home/user/.local/bin:$PATH"

# Setup Direktori Kerja
WORKDIR /app

# Install Dependencies
COPY --chown=user ./requirements.txt requirements.txt
RUN pip install --no-cache-dir --upgrade -r requirements.txt

# Copy File Aplikasi & Artefak Model
COPY --chown=user . /app

# Expose Port & Jalankan Server
EXPOSE 7860
CMD ["python", "app.py"]



### 4. Daftar Dependensi (`requirements.txt`)

In [3]:
req_path = os.path.join("daynight-project", "requirements.txt")
with open(req_path, "r", encoding="utf-8") as f:
    req_content = f.read()

print("Daftar requirements.txt:")
print(req_content)


Daftar requirements.txt:
flask
tensorflow-cpu
numpy
scikit-learn
scikit-image
opencv-python-headless
gunicorn



### 5. Prosedur Deployment ke Hugging Face Spaces (Panduan CLI)
Langkah-langkah yang dijalankan melalui Git CLI:
1. Buat Space baru di [Hugging Face Spaces](https://huggingface.co/spaces) dengan memilih **Docker SDK**.
2. Buat **Access Token (Write)** pada profil Hugging Face Settings.
3. Jalankan perintah Git:
   ```bash
   git clone https://huggingface.co/spaces/NAMA_USER/daynight-classifier
   cd daynight-classifier
   # Salin app.py, Dockerfile, requirements.txt, day_night_model.h5, scaler.pkl ke folder ini
   git add .
   git commit -m "Deploy model Day vs Night ke Hugging Face"
   git push
   ```
4. Masukkan username Hugging Face dan Access Token saat diminta autentikasi.
5. Hugging Face Spaces secara otomatis akan mem-build container Docker dan menjalankan aplikasi di port 7860 dengan status **Running**.


### Kesimpulan Tugas Praktikum JS15
1. **Pipeline Terpadu**: Praktikum ini menyatukan seluruh mata rantai *Machine Learning Operations* (MLOps), mulai dari tahap ekstraksi fitur HOG, normalisasi data, pelatihan neural network, serialisasi model (.h5) dan scaler (.pkl), hingga pembuatan aplikasi web Flask.
2. **Containerization dengan Docker**: Penggunaan Docker menjamin portabilitas tinggi di mana aplikasi dapat berjalan secara identik di lingkungan lokal maupun server cloud Hugging Face tanpa kendala perbedaan versi pustaka.
3. **Kesiapan Produksi**: Pola pipeline deployment ini dapat langsung diterapkan pada dataset dan domain proyek lain (misalnya prediksi burnout karyawan pada studi kasus berikutnya).
